# 00 – Kiểm tra môi trường (máy cá nhân và Colab)

Notebook chạy thử code Phần 2 (cấu hình, chỉ số, ghi log) trên **dữ liệu giả lập**, không cần dữ liệu Chicago.
Mục đích: xác nhận repo chạy được trên Colab mà không phải sửa code.

Mở trên Colab: https://colab.research.google.com/github/tanbentam/KhaiPhaDuLieu/blob/main/notebooks/00_kiem_tra_moi_truong.ipynb

Cách chạy: chọn Runtime > Run all (Colab) hoặc Run All (Jupyter/VS Code). Kết quả mong đợi: ô cuối in `TAT CA DAT`.

**Ô thiết lập ngay dưới đây phải là ô đầu tiên của mọi notebook trong đồ án.**

In [ ]:
# ===== O THIET LAP (chep nguyen o nay vao dau moi notebook) =====
# Colab: clone repo, cai thu vien, them goc repo vao sys.path.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Du lieu that tren Colab: bo comment 2 dong duoi khi nhom da chot thu muc Drive
# from src.colab import link_drive_data
# link_drive_data("KhaiPhaDuLieu")

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## 1. Phiên bản thư viện và máy

In [ ]:
from src.evaluation.runlog import lib_versions, machine_info, git_commit
print("May      :", machine_info())
print("Thu vien :", lib_versions())
print("Commit   :", git_commit(ROOT))

## 2. Đọc cấu hình

Log và nhãn của lần thử được ghi vào `outputs/_test_env/` và bị xoá ở ô cuối.

In [ ]:
from src.config import load_config
cfg = load_config()
cfg["paths"]["run_log"] = "outputs/_test_env/run_log.csv"
cfg["paths"]["labels_dir"] = "outputs/_test_env/labels"
print("seed:", cfg["project"]["seed"], "| crs:", cfg["project"]["crs"], "| lat cat:", list(cfg["time_slices"]))

## 3. Chạy thử K-Means và DBSCAN trên dữ liệu giả lập

4 cụm cách nhau 3000 m (giống toạ độ mét EPSG:26916) và 10% điểm nhiễu rải đều. `trip_id` có tiền tố `TEST_`.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import DBSCAN, KMeans
from src.evaluation import run_and_log, read_run_log

rng = np.random.default_rng(cfg["project"]["seed"])
Xb, _ = make_blobs(9000, centers=[[0, 0], [3000, 0], [0, 3000], [3000, 3000]], cluster_std=200,
                   random_state=cfg["project"]["seed"])
X = np.vstack([Xb, rng.uniform(-1000, 4000, size=(1000, 2))])
trip_ids = np.array([f"TEST_{i:05d}" for i in range(len(X))])

p_db = {"eps": 150, "min_samples": 10}
row_db = run_and_log(cfg, "dbscan", lambda X: DBSCAN(**p_db).fit_predict(X), X, trip_ids,
                     member="M2", time_slice="WEEKDAY_AM", sample_id="TEST_synthetic", params=p_db,
                     note="TEST notebook 00")
p_km = {"n_clusters": 4, "init": "k-means++", "n_init": 10, "random_state": cfg["project"]["seed"]}
row_km = run_and_log(cfg, "kmeans", lambda X: KMeans(**p_km).fit_predict(X), X, trip_ids,
                     member="M2", time_slice="WEEKDAY_AM", sample_id="TEST_synthetic", params=p_km,
                     note="TEST notebook 00")

log = read_run_log(cfg["_root"] / cfg["paths"]["run_log"])
log[["run_id", "n_clusters", "noise_ratio", "silhouette", "davies_bouldin",
     "runtime_fit_sec", "mem_delta_mb", "machine"]]

## 4. Kiểm tra kết quả và dọn dẹp

In [ ]:
import shutil, pandas as pd

lab = pd.read_parquet(cfg["_root"] / cfg["paths"]["labels_dir"] / f"{row_db['run_id']}.parquet")
checks = {
    "log co 2 dong": len(log) == 2,
    # 4 cum gia lap deu phai duoc tim thay (co the them cum nho do nhieu ngau nhien day)
    "DBSCAN tim du 4 cum lon (>= 1000 diem)": (lab.loc[lab["label"] >= 0, "label"].value_counts() >= 1000).sum() == 4,
    "DBSCAN co nhieu (noise_ratio > 0)": row_db["noise_ratio"] > 0,
    "K-Means khong co nhieu": row_km["noise_ratio"] == 0,
    "silhouette trong [-1, 1]": -1 <= row_db["silhouette"] <= 1,
    "file nhan du 10000 dong, dung thu tu": len(lab) == 10000 and (lab["trip_id"].values == trip_ids).all(),
    "runtime > 0": row_db["runtime_fit_sec"] > 0,
}
for k, v in checks.items():
    print("DAT  " if v else "LOI  ", k)

shutil.rmtree(cfg["_root"] / "outputs" / "_test_env", ignore_errors=True)
outputs = cfg["_root"] / "outputs"
if outputs.is_dir() and not any(outputs.iterdir()):
    outputs.rmdir()
print()
print("TAT CA DAT" if all(checks.values()) else "CO LOI - xem dong LOI o tren")